# Task 2: Toxicity Classification (Colab runner)
Jalankan sel berurutan. Run yang sudah `DONE` dilewati, jadi aman diulang setelah runtime putus.

In [ ]:
# Setup: Colab + Drive. Jigsaw (train.csv, test.csv, test_labels.csv) taruh di Drive
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/ivant8k/IF5153-Chat-Moderation.git"
BRANCH = "feat/task-2"
DRIVE_JIGSAW = "/content/drive/MyDrive/tubes_nlp/jigsaw"      # folder berisi 3 csv Jigsaw
DRIVE_OUT = "/content/drive/MyDrive/tubes_nlp/task_2_runs"    # checkpoint + hasil (awet)

from google.colab import drive
drive.mount("/content/drive")
if not Path("/content/repo").exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, "/content/repo"], check=True)
os.chdir("/content/repo")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "scikit-learn", "pandas"], check=True)
Path("data/raw").mkdir(parents=True, exist_ok=True)
if not Path("data/raw/conda").exists():
    subprocess.run("git clone --depth 1 https://github.com/usydnlp/CONDA.git data/raw/conda && rm -rf data/raw/conda/.git", shell=True, check=True)
if not Path("data/raw/jigsaw").exists():
    os.symlink(DRIVE_JIGSAW, "data/raw/jigsaw")
Path(DRIVE_OUT).mkdir(parents=True, exist_ok=True)
Path("outputs/task_2").mkdir(parents=True, exist_ok=True)
if not Path("outputs/task_2/runs").exists():
    os.symlink(DRIVE_OUT, "outputs/task_2/runs")
!nvidia-smi | head -15

## 1. Data
Jigsaw + CONDA (split Task 1) + NoisyHate -> `data/processed/task_2`

In [ ]:
!python src/task_2/prepare.py

## 2. Baseline
E0 TF-IDF + LR dan zero-shot `unitary/toxic-bert`

In [ ]:
!python src/task_2/baseline.py --run b0_tfidf
!python src/task_2/evaluate.py --run b0_tfidf
!python src/task_2/evaluate.py --run z_toxicbert --kind zeroshot

## 3. Arm DistilRoBERTa
E1 Jigsaw saja, E2 campuran (kandidat utama), E3 bertahap (Jigsaw lalu CONDA)

In [ ]:
!python src/task_2/train.py --run e1_jigsaw --sources jigsaw
!python src/task_2/evaluate.py --run e1_jigsaw

In [ ]:
!python src/task_2/train.py --run e2_mix --sources jigsaw,conda
!python src/task_2/evaluate.py --run e2_mix

In [ ]:
!python src/task_2/train.py --run e3_seq --sources conda --init-from e1_jigsaw --lr 1e-5 --select all
!python src/task_2/evaluate.py --run e3_seq

## 4. Ringkasan

In [ ]:
!python src/task_2/summary.py

## 5. Demo kontrak output (ganti run sesuai pemenang)

In [ ]:
import sys
sys.path.insert(0, "src/task_2")
from predict import ToxicityClassifier
import json
clf = ToxicityClassifier("e2_mix", mode="balanced")
for r in clf.classify(["you are idiot", "gg wp", "report mid he is feeding", "u | r | trash | noob"]):
    print(json.dumps(r, indent=1))